# Can a category rule teach a neural model?

**CS 691 · Stage II · Extension after Modules 07 and 08**

Ava returns a camera. Its catalog record should say both *camera* and *equipment*, because every camera belongs to the equipment category. Suppose two separate scores from the same photograph disagree. Can we teach a model this relationship when some catalog labels are missing?

In Module 07 you tried five values for a single weight. Here you will fit six numbers at once, use a rule to influence that fit, and compare two models on items neither saw during training. You will also try a hard rule after scoring, as in Module 08.

Use a Python notebook runtime with PyTorch. A standard Google Colab Python runtime includes it. Run the cells from top to bottom on CPU. Every feature, label and item in this lab is fictional.

In [1]:
import torch
from torch import nn
import torch.nn.functional as F

torch.set_num_threads(1)

## 1 · Read the evidence

Each item has two numbers between 0 and 1 that describe its photograph. The first measures a lens-like visual cue and the second a support-frame cue. We supply those measurements so we can study what happens **after** feature extraction. A label of 1 means the cataloger recorded the category as present. A label of 0 means the cataloger explicitly recorded it as absent. A question mark means the label was left blank.

Some rows for cameras lack an equipment label. The tripods show us why the category relationship points in only one direction. A tripod is equipment and is not a camera.

**Predict.** If a cataloger marked C1 as a camera and left its equipment field blank, which target value should the equipment output copy during the ordinary label-matching step?

In [2]:
# Each row contains ID, two cues, then camera and equipment labels.
training = [
    ("C1",  (1.00, .00), (1, None)),
    ("C2",  ( .85, .10), (1, None)),
    ("T1",  ( .00, 1.0), (0, 1)),
    ("T2",  ( .10, .90), (0, 1)),
    ("P1",  ( .00, .00), (0, 0)),
    ("P2",  ( .10, .05), (0, 0)),
]
for item, features, labels in training:
    shown = ["?" if label is None else str(label) for label in labels]
    print(item, *features, *shown)

x_train = torch.tensor([row[1] for row in training])
observed = torch.tensor([[label is not None for label in row[2]]
                         for row in training])
# Zero fills the rectangular array. The mask excludes those filler entries.
targets = torch.tensor([[float(label) if label is not None else 0.0
                         for label in row[2]] for row in training])
print("Observed labels:", int(observed.sum()), "of", observed.numel())

C1 1.0 0.0 1 ?
C2 0.85 0.1 1 ?
T1 0.0 1.0 0 1
T2 0.1 0.9 0 1
P1 0.0 0.0 0 0
P2 0.1 0.05 0 0
Observed labels: 10 of 12


### Check your prediction

<details><summary>Reveal the worked answer</summary>

Skip C1's blank equipment label when computing the data loss. A zero there would teach an explicitly negative label. The rule will supply separate information.

</details>

A two-output neural layer gives each category its own weights for the two cues and its own bias, a starting offset. This makes six adjustable values. For each category it computes a weighted sum plus its bias. This raw result is called a **logit**. The sigmoid function turns a logit into a score between 0 and 1. A logit of zero becomes a score of 0.5. We use 0.5 as the inclusive cutoff for a positive prediction in this lab.

In [3]:
def fresh_model():
    model = nn.Linear(2, 2)
    # Explicit starting values make both conditions identical.
    with torch.no_grad():
        model.weight.zero_()
        model.bias.zero_()
    return model

start = fresh_model()
with torch.no_grad():
    raw = start(x_train[:1])
    print("C1 starting logits:", raw[0].tolist())
    print("C1 starting scores:", torch.sigmoid(raw)[0].tolist())

C1 starting logits: [0.0, 0.0]
C1 starting scores: [0.5, 0.5]


## 2 · Give a rule a training cost

The ordinary data loss rewards high scores on observed positive labels and low scores on observed negative labels. We use binary cross-entropy. For example, for a positive label, the loss is the negative natural logarithm of its score. Raising that score from 0.5 to 0.9 reduces the loss from about 0.69 to 0.11. PyTorch calculates it directly from logits for numerical stability.

The desk's category rule says that if an item is a camera, it is equipment. As a training preference we want the equipment score to be at least the camera score. For each item, take the camera score minus the equipment score. Keep the positive part and square it. Average those values across the training items to get the **rule penalty**.

**Predict.** One item scores 0.80 for camera and 0.30 for equipment. A second scores 0.20 and 0.90. Which item adds to the penalty? How much does it add before averaging?

In [4]:
def row_penalties(scores):
    camera = scores[:, 0]       # All rows, first output
    equipment = scores[:, 1]    # All rows, second output
    shortfall = (camera - equipment).clamp(min=0)
    return shortfall.square()

example = torch.tensor([[.8, .3], [.2, .9]])
penalties = row_penalties(example)
print("Row penalties:", [round(v, 3) for v in penalties.tolist()])
print("Average:", round(penalties.mean().item(), 3))

Row penalties: [0.25, 0.0]
Average: 0.125


### Check your prediction

<details><summary>Reveal the worked answer</summary>

The first item contributes (0.80 − 0.30)² = 0.25 and the second contributes zero, giving a mean of 0.125. An equipment score above the camera score is consistent with the stated hierarchy. A rule-consistent pair can still predict the wrong real-world category. The reference labels let us check that separately.

</details>


## 3 · Fit two models under the same conditions

The **objective** is data loss plus a chosen multiplier times the rule penalty. Set the multiplier to 0 for the labels-only model and to 4 for the model trained with the rule. Four is an illustrative choice fixed before evaluation.

Each training step computes the objective for all six training items. PyTorch's backward operation gives the derivative of that objective with respect to each weight and bias. Think of a derivative as the slope of the loss when you nudge one parameter. The optimizer subtracts a small multiple of that slope from the parameter. This is how the program seeks a smaller loss without listing every candidate set of six weights.

Both conditions use the same six records, starting parameters, learning rate and number of updates. Only the multiplier differs. The held-out items and their labels play no part in fitting.

In [5]:
def fit(rule_strength):
    model = fresh_model()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
    for step in range(400):
        optimizer.zero_grad()         # Clear derivatives from the previous step.
        logits = model(x_train)
        data_loss = F.binary_cross_entropy_with_logits(
            logits[observed], targets[observed])
        rule_loss = row_penalties(torch.sigmoid(logits)).mean()
        objective = data_loss + rule_strength * rule_loss
        objective.backward()
        optimizer.step()
    return model

labels_only = fit(0)
with_rule = fit(4)

## 4 · Inspect the held-out predictions

Here are five new catalog items that have independently specified reference labels in this toy dataset. C3 and C4 are cameras, T3 is a tripod, P3 is a plain card, and P4 is a card printed with a camera icon. P4 deliberately looks like a camera to our two-cue representation.

**Predict.** On the true cameras, which score might rise when the rule is included? On the printed card, could that same change hurt? Run the comparison and inspect one row before reading the totals.

The first reference number is camera and the second is equipment. The program prints scores to two decimals for reading. Decisions use the full scores.

In [6]:
held_out = [
    ("C3", (.95, .05), (1, 1)),
    ("C4", (.75, .12), (1, 1)),
    ("T3", (.05, .82), (0, 1)),
    ("P3", (.08, .03), (0, 0)),
    ("P4", (.86, .04), (0, 0)),
]
x_test = torch.tensor([row[1] for row in held_out])
truth = torch.tensor([row[2] for row in held_out], dtype=torch.bool)

@torch.no_grad()
def evaluate(model):
    scores = torch.sigmoid(model(x_test))
    decisions = scores >= 0.5
    return scores, decisions

base_scores, base_predictions = evaluate(labels_only)
rule_scores, rule_predictions = evaluate(with_rule)
print("Item  reference   labels only       with rule")
print("      cam equip   cam   equip       cam   equip")
for i, (item, _, label) in enumerate(held_out):
    c0, e0 = base_scores[i].tolist()
    c1, e1 = rule_scores[i].tolist()
    print(f"{item:4}   {label[0]}    {label[1]}    {c0:.2f}  {e0:.2f}        {c1:.2f}  {e1:.2f}")

Item  reference   labels only       with rule
      cam equip   cam   equip       cam   equip
C3     1    1    0.94  0.09        0.91  0.87
C4     1    1    0.82  0.13        0.76  0.81
T3     0    1    0.02  0.88        0.02  0.89
P3     0    0    0.10  0.09        0.10  0.17
P4     0    0    0.91  0.09        0.87  0.81


### Count different kinds of success

Category accuracy counts matches to the reference label for each category separately. A **hard violation** is an item predicted to be a camera but predicted to be outside equipment. The mean squared **score shortfall** can also detect a smaller disagreement even if both scores fall on the same side of 0.5. These measurements answer different questions.

In [7]:
def report(name, scores, predictions):
    correct = (predictions == truth).sum(dim=0)
    hard = predictions[:, 0] & ~predictions[:, 1]
    print(f"{name}: camera {int(correct[0])}/5, equipment {int(correct[1])}/5,"
          f" hard violations {int(hard.sum())}/5,"
          f" mean squared shortfall {row_penalties(scores).mean().item():.4f}")

report("Labels only", base_scores, base_predictions)
report("With rule", rule_scores, rule_predictions)

Labels only: camera 4/5, equipment 3/5, hard violations 3/5, mean squared shortfall 0.3743
With rule: camera 4/5, equipment 4/5, hard violations 0/5, mean squared shortfall 0.0010


## 5 · Enforce the rule after prediction

The training penalty was soft. A hard decoding policy can require every positive camera prediction to carry a positive equipment prediction. One possible policy copies a positive camera decision into the equipment decision. Keep each camera decision as it was. In the code, the vertical bar means logical OR, true when either of its inputs is true.

**Predict.** If the model wrongly calls the printed card a camera, can this hard policy repair the camera mistake? Check every changed decision on the labels-only model.

In [8]:
def close_upward(predictions):
    repaired = predictions.clone()
    repaired[:, 1] = predictions[:, 1] | predictions[:, 0]
    return repaired

repaired = close_upward(base_predictions)
def show(bits):
    return tuple(int(v) for v in bits.tolist())
for i, (item, _, _) in enumerate(held_out):
    if not torch.equal(repaired[i], base_predictions[i]):
        print(item, show(base_predictions[i]), "->", show(repaired[i]),
              "reference", show(truth[i]))
hard_violations = repaired[:, 0] & ~repaired[:, 1]
assert not hard_violations.any()
print("Hard violations after decoding:", int(hard_violations.sum()))

C3 (1, 0) -> (1, 1) reference (1, 1)
C4 (1, 0) -> (1, 1) reference (1, 1)
P4 (1, 0) -> (1, 1) reference (0, 0)
Hard violations after decoding: 0


## Check your understanding

1. Why does the equipment result improve overall even though the printed card's equipment prediction gets worse?
2. Can a model have zero hard violations on these five items and still have a positive score penalty?
3. What additional test would distinguish a detector that recognizes physical cameras from one that responds to printed camera icons?

<details><summary>Reveal the worked answers</summary>

1. The rule-trained model fixes the equipment label on two real cameras and introduces one equipment error on P4. Equipment accuracy moves from 3/5 to 4/5. Camera accuracy stays at 4/5. On P4 the misleading first cue raises the camera score, and learning the category relationship also raises its equipment score.
2. Yes. On C3 the rule-trained model gives about 0.91 for camera and 0.87 for equipment. Both pass 0.5, so there is no hard violation. The positive difference still incurs a small penalty. Zero hard violations here describes five test items. The hard decoding rule enforces the implication for any pair of input decisions.
3. Set aside examples of physical cameras and photographs of camera icons on cards or packaging. Label the groups independently. Compare false camera detections and the equipment errors they trigger in each group. A result on five constructed records cannot estimate performance on a real inventory.

</details>

**Try a variation.** Choose a new rule multiplier, write down your prediction first, and repeat the comparison. Record which individual decisions changed. Once a test set helps you choose a multiplier, it has become development data. Evaluate the chosen model on new examples to make a fresh performance claim.

### PyTorch reference

The library documents how cross-entropy combines sigmoid and label loss and how SGD updates the parameters using gradients.

[PyTorch binary cross-entropy with logits](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html) · [PyTorch SGD](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html)